# DECORE + Quantization — VGG-16 / CIFAR-10

Combines DECORE (RL **pruning**) with **INT8 post-training quantization**, comparing four variants:

1. **baseline** — full VGG-16 (fp32)
2. **baseline + INT8** — full model, post-training static INT8
3. **decored** — DECORE-pruned (fp32)
4. **decore + INT8** — pruned model, post-training static INT8

Pruning removes redundant channels; INT8 then stores the rest at 8-bit. The two are
multiplicative, so `decore + INT8` is roughly an order of magnitude smaller than the
original VGG-16 at essentially the same accuracy.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import numpy as np, torch
import onnxruntime as ort
from decore.data import build_loaders
from decore.models import build_model
from decore.metrics import count_parameters
from decore.engine import evaluate
from decore.quantize import quantize_onnx_int8
from scripts.export_onnx import _export  # torch->onnx single-file export helper

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('device:', device)
train_loader, test_loader = build_loaders('data', 128, 0)
results = {}  # variant -> dict(acc, size_mb, note)

## 1 & 3. Baseline and DECORE-pruned (fp32)

In [ ]:
baseline = build_model('vgg16', 10, device)
baseline.load_state_dict(torch.load('runs/baseline.pth', map_location=device), strict=False)
pruned = torch.load('runs/pruned_vgg16_l50.pt', map_location=device, weights_only=False).to(device)

acc_b = evaluate(baseline, test_loader, device)[0]
acc_p = evaluate(pruned, test_loader, device)[0]
results['baseline'] = dict(acc=acc_b, size_mb=count_parameters(baseline)*4/1e6, note='fp32')
results['decored']  = dict(acc=acc_p, size_mb=count_parameters(pruned)*4/1e6, note='fp32, pruned')
print(f"baseline  {acc_b:.2f}%  {results['baseline']['size_mb']:.1f}MB")
print(f"decored   {acc_p:.2f}%  {results['decored']['size_mb']:.1f}MB")

## 2 & 4. INT8 (post-training static quantization)

Export each model to ONNX, then statically quantize to INT8 with a small CIFAR calibration pass
(needed so conv-layer activations can be int8). Evaluate the INT8 models with ONNX Runtime.

In [ ]:
os.makedirs('web/models', exist_ok=True)
_export(baseline.to(device).eval(), 'web/models/baseline.onnx')
_export(pruned.to(device).eval(),   'web/models/pruned.onnx')
quantize_onnx_int8('web/models/baseline.onnx', 'web/models/baseline.int8.onnx', num_calib=200)
quantize_onnx_int8('web/models/pruned.onnx',   'web/models/pruned.int8.onnx',   num_calib=200)

def eval_onnx(path):
    s = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    c = t = 0
    for x, y in test_loader:
        out = s.run(None, {'input': x.numpy().astype(np.float32)})[0]
        c += (out.argmax(1) == y.numpy()).sum(); t += len(y)
    return 100.0 * c / t

for tag, mdl in [('baseline + int8', 'baseline'), ('decore + int8', 'pruned')]:
    p = f'web/models/{mdl}.int8.onnx'
    results[tag] = dict(acc=eval_onnx(p), size_mb=os.path.getsize(p)/1e6, note='static INT8 (ONNX)')
    print(f"{tag:16s} {results[tag]['acc']:.2f}%  {results[tag]['size_mb']:.1f}MB")

## Comparison

In [ ]:
order = ['baseline', 'baseline + int8', 'decored', 'decore + int8']
base_mb = results['baseline']['size_mb']
print(f"{'variant':<20}{'top-1':>8}{'size (MB)':>12}{'vs baseline':>13}   note")
print('-' * 74)
for k in order:
    r = results[k]
    print(f"{k:<20}{r['acc']:>7.2f}%{r['size_mb']:>11.1f}M{base_mb/r['size_mb']:>11.1f}x   {r['note']}")